对称加密 - 加密解密用同一个key

特点：速度快 （基于AES算法，如果用了复杂算法，对称加密也会很慢）
适用场景：大量数据加密
算法：AES

非对称加密 - 公钥加密，私钥解密
适用场景：密钥交换、数字签名  -- why？
算法：RSA


In [ ]:
# 对称加密

from cryptography.fernet import Fernet

# 生成key - 二进制的
# generate_key 是Fernet类的静态方法 cryptography 提供的对称加密规范
fixed_key = Fernet.generate_key()
print(type(fixed_key))

# 创建加密对象
cipher = Fernet(fixed_key)

# 加密
send_message = "请加密"
print(type(send_message)) # str 字符串对象

# encrypt 要求输入bytes，所以这里需要对send_message 进行encode后 再加密
encrypted = cipher.encrypt(send_message.encode()) 
print(f"加密后: {encrypted}, type is {type(encrypted)}")

# 解密
decrypted = cipher.decrypt(encrypted)
received_message = decrypted.decode() # bytes -> str
print(f"解密后: {received_message}")

<class 'bytes'>
<class 'type'>
加密后: b'gAAAAABqMBU4tgEg03M05rXEIQqshQ5IZxUVi-XPdnL_l2ASVxCOZ-A7sZeMAVvoT_fqiAYA1aiAVOec-oTUiprEgC3K2WGt5g=='
解密后: 请加密


In [ ]:
# 非对称加密
from cryptography.hazmat.primitives.asymmetric import rsa,padding
from cryptography.hazmat.primitives import hashes

# 利用rsa算法 生成密钥对
private_key = rsa.generate_private_key(
    public_exponent=65537, # rsa的公开指数 65537 是业界标准值（HTTPS，SSH，JWT，数字签名都用）
    key_size=2048 # 密钥长度 位数越大越安全，但是会慢
)

# 从私钥提取公钥
public_key = private_key.public_key()

# 公钥加密
s_message = "非对称加密"

# 明文 -> RSA公钥 -> 密文
encrypted = public_key.encrypt(
    s_message.encode(),
     # padding - 随机填充 避免明文被有规律的加密的风险
    padding.OAEP(  # OAEP是RSA推荐标准
 
        # mgf 掩码生成函数 - 增加随机性，作用是相同的明文每次加密的结果不同
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # 计算哈希 SHA256 -> output: 256位 32字节
        algorithm=hashes.SHA256(),
        label=None # 额外标签，方便携带一些业务标识、上下文标识
    )
)
print(f"加密后: {encrypted[:30]}...")

# 私钥解密
decrypted = private_key.decrypt(
    encrypted,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()), # 必须和加密一致
        algorithm=hashes.SHA256(),
        label=None
    )
)

print(f"解密后: {decrypted.decode()}") 

加密后: b'\x1c\x04\x92\x9b{9\x98\x189*r~\xac\x1c{\xc2*R\xe9F{;\xd7{"!\xa0>\xa1p'...
解密后: 非对称加密


总结：
1. 对称加密是加密和解密用同一个key ； 非对称加密是公钥加密，私钥解密。
2. 对称加密适用于大量数据加密（依据算法来决定），非对称加密适用于 密钥交换、数字签名。
3. 密钥交换的过程 - A发送给B一个公钥（“锁”） -> B 拿到后用公钥给 对称密钥加密 并“加锁”发送给A -> A拿到后用私钥（“钥匙”）解锁，并用对称加密的key解密。  整个过程中，A的私钥没有被传输，所以即便内容和公钥和对称加密的内容被截胡，但是没有私钥一般很难解锁。
4. 对称加密我们之前常用于自动化测试的测试账户密码加密。
5. 对称加密不仅有以上使用过的AES这一种算法还有其他。大部分都适用于大量数据加密，运算效率高，资源消耗低CPU友好
   非对称也是不仅有RSA，还有其他。但是也都不太适用于大量数据，比较适用于密钥交换，数字签名。 
   所以市面上很多流行写法是混合加密。

6. OAEP 和 MGF1 也并非 非对称加密固定的方式，它们的关系是 
OAEP = 装修方案 -> 我要填充
MGF1 = 装修工人 -> 操作填充
SHA256 = 工人使用的工具 -> 哈希算法（哈希不是加密，是不可逆计算）
随机数 = 原材料 -> 用哪些数字加密后填充

下面的题目均需要通过-对称加解密-实现

1. 对 "hello word"进行加密，比如加密得到的数据为 "adsfjadsfjadjfasdkjfaskd"
2. 将得到的加密数据，在随机位置插入一些字符，比如在第二位插入数字2024，就得到"a2024dsfjadsfjadjfasdkjfaskd"
3. 将第二步得到的数据再进行加密。比如得到再次加密的数据为 "fajdfjadsfjadskjfadskjfadsjfakdskf"
4. 依次进行解密，最终解密的数据要和之前的 "hello word"一致。

In [15]:
from cryptography.fernet import Fernet

# 生成key - 二进制
key = Fernet.generate_key()

# 创建加密对象
cipher = Fernet(key)

ori_message = b"hello world" # -> 转化为byte

# 第一次加密
first_encrypted = cipher.encrypt(ori_message)
print(f"Fisrt time encrypt complete {first_encrypted}")

# 对需要插入的内容encode
insert_data = b"2024"
l = len(insert_data)

# 拼接
temp_message = first_encrypted[:1] + insert_data + first_encrypted[1:]
print(f"mid_message is {temp_message}")

# 二次加密
second_encrypted = cipher.encrypt(temp_message)
print(f"Secomd time encrypt complete {second_encrypted}")

# 第一轮解密
second_decrypted = cipher.decrypt(second_encrypted)
mid_r_message = second_decrypted.decode()
print(f"After the first decrypted {mid_r_message}")

# 删除插入的内容
restored_first_encrypyed = second_decrypted[:1] + second_decrypted[l+1:]

# 第二次解密
final_encrypted = cipher.decrypt(restored_first_encrypyed)
final_message = final_encrypted.decode()
print(f"After the second decrypted {final_message}")

Fisrt time encrypt complete b'gAAAAABqMDGubhK7XJkp6mkOGK7e65WvCpiadZ4oBZFb6lOHDtPuf0w3qjGXOe3mXEBcPR4FEcf1lez7Cbd5cZEpNpcpUb3bRg=='
mid_message is b'g2024AAAAABqMDGubhK7XJkp6mkOGK7e65WvCpiadZ4oBZFb6lOHDtPuf0w3qjGXOe3mXEBcPR4FEcf1lez7Cbd5cZEpNpcpUb3bRg=='
Secomd time encrypt complete b'gAAAAABqMDGuJ2WkcWrjheSuBmZIEM-NMZErfka9vNOm3V2X8NGloTd5kBT-3KiUv3Lg4GR-aHRZUrAyhON7kfVaBEh3xilJZ3n7klICqC9dl_crWK_-G5z0Gd4LrpzI7ioFCaVRvvaObgCuGYRiu_aPCdcO7XajfeEpVnBRzFy6IJUrWYUX98rfmKNM66S1Tb0vlCGmGkqhf6ImUafvEyx5fOCp79hflQ=='
After the first decrypted g2024AAAAABqMDGubhK7XJkp6mkOGK7e65WvCpiadZ4oBZFb6lOHDtPuf0w3qjGXOe3mXEBcPR4FEcf1lez7Cbd5cZEpNpcpUb3bRg==
After the second decrypted hello world
